In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.3 Dirac Notation, Bases, and Spectral Decomposition

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.3",
    title="Dirac Notation, Bases, and Spectral Decomposition",
    blurb="The language. We give the inner product and the operators of the last "
    "two notebooks their proper notation — kets, bras, and the outer product — and "
    "find it is not shorthand but a calculus. Projectors extract components, the "
    "resolution of the identity lets us expand in any basis at will, and the spectral "
    "decomposition turns any function of an operator into a function of its "
    "eigenvalues. With this in hand, the physics can begin.",
    difficulty="advanced",
    estimate="150–190 min",
)

## Notebook overview

Two notebooks ago we built the space; one notebook ago we built the operators. This notebook builds
the **language** in which the rest of the volume is written — Dirac's notation of kets and bras — and
the surprise it holds is that the notation is not a convenience laid on top of the mathematics but a
*calculus*: a small set of symbols whose manipulation rules do the work for us. It is the closing
notebook of Movement 0, the mathematical arsenal, and by its end the formalism is complete and the
physics can begin.

The notebook also keeps two promises. In [§6.1](complex-vector-spaces.ipynb) we used $\langle u|v\rangle$ as shorthand and promised
that the **bra** $\langle u|$ would later become an object in its own right; here it does — a *dual
vector*, the linear functional "take the inner product with $u$." In [§6.2](operators-spectral-theorem.ipynb) we wrote the spectral
theorem as a matrix factorization $A=V\Lambda V^{\dagger}$ and promised an outer-product form; here
it arrives as $A=\sum_\lambda\lambda\,|\lambda\rangle\langle\lambda|$, a Hermitian operator written
as its eigenvalues times the **projectors** onto its eigenvectors. After this notebook there are no
remaining notation deferrals in the arsenal.

Three constructions carry the whole calculus, and the notebook is built around them. The **projector**
$|e\rangle\langle e|$ extracts the component of a state along $|e\rangle$. The **resolution of the
identity** $\sum_i|e_i\rangle\langle e_i|=I$ is the compact statement that any state expands in any
orthonormal basis — and the single most useful move in the formalism is to *insert the identity*
anywhere a change of basis is wanted. The **spectral decomposition** $A=\sum_i\lambda_i|\lambda_i
\rangle\langle\lambda_i|$ is "diagonalize" in its cleanest dress, and it makes *any function of an
operator* trivial: apply the function to the eigenvalues. That last fact is the general machinery
behind the exponential map of [§6.2](operators-spectral-theorem.ipynb) and behind the time-evolution operator $U(t)=e^{-iHt/\hbar}$ that will
run all of dynamics ([§6.7](time-evolution.ipynb)).

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts that name the exact operation to run — `numpy.outer` (with `numpy.conj` for the bra) to
build $|u\rangle\langle v|$, explicit projector and resolution-of-identity construction,
`numpy.linalg.eigh` for the spectral data, `scipy.linalg.expm` to cross-check the spectral
exponential — so the method is never something to reverse-engineer.

> **A word on what closes here.** This notebook *resolves* the two boundaries the arsenal carried.
> The bra $\langle u|$ is now a dual vector (as promised in [§6.1](complex-vector-spaces.ipynb)), and the outer product, projectors, the
> projector form of the spectral theorem, and functions of operators are all developed (as promised in
> [§6.2](operators-spectral-theorem.ipynb)). What we deliberately do *not* do is the physics: the projector *is* the mathematics of a
> measurement, but the measurement postulate itself — outcomes, probabilities, collapse — is [§6.5](postulates.ipynb). We
> build the apparatus here and let the physics out in the next movement.
>
> **How to read the checks.** Each exercise closes with a `validate` call against an independent
> fact: the outer product acting as an operator; a projector's idempotence, Hermiticity, and rank;
> the resolution of the identity to machine precision; a change of basis preserving eigenvalues and
> inner products; the spectral decomposition reconstructing a Hermitian operator from its
> eigen-projectors; a function of an operator matching `scipy.linalg.expm` and a square root squaring
> back; and a qubit's outcome-weights summing to one in two different bases. A ✓ is strong evidence;
> a ✗ is a prompt to *locate the discrepancy*.
>
> **Scope.** Finite-dimensional, orthonormal bases throughout. The measurement postulate is [§6.5](postulates.ipynb); the
> Stern–Gerlach experiment and the two-state system that open the physics are [§6.4](stern-gerlach-qubit.ipynb); the
> time-evolution operator is [§6.7](time-evolution.ipynb); the Bloch sphere is [§6.8](bloch-sphere-entanglement.ipynb). See Sakurai & Napolitano (Ch. 1); Dirac,
> *The Principles of Quantum Mechanics*; and Notebooks [§6.1](complex-vector-spaces.ipynb) (the inner product), [§6.2](operators-spectral-theorem.ipynb) (the spectral
> theorem, the exponential map).

## Theory in brief

### Kets, bras, and the dual space

A **ket** $|\psi\rangle$ is a state vector, a column of complex numbers. A **bra** $\langle\phi|$ is
the **linear functional** that eats a ket and returns the number $\langle\phi|\psi\rangle$ — "take
the inner product with $\phi$." The bras form the **dual space**; in components a bra is the
conjugate-transpose row vector, the adjoint of the ket,

```{math}
:label: eq-bra-ket
|\psi\rangle=\begin{pmatrix}\psi_1\\\vdots\\\psi_n\end{pmatrix},\qquad \langle\phi|=(\phi_1^{*},\dots,\phi_n^{*}),\qquad \langle\phi|\psi\rangle=\sum_i\phi_i^{*}\psi_i .
```

This is the meaning deferred from [§6.1](complex-vector-spaces.ipynb): the inner product is a bra acting on a ket.

### The outer product

Multiplied the *other* way, $|u\rangle\langle v|$ is an **operator**. Acting on a ket it gives

```{math}
:label: eq-outer
\big(|u\rangle\langle v|\big)\,|w\rangle=\langle v|w\rangle\,|u\rangle ,
```

"measure the overlap with $v$, output that much of $u$." In components it is `numpy.outer(u,
numpy.conj(v))`. The inner product $\langle v|u\rangle$ is a *number*; the outer product $|u\rangle
\langle v|$ is an *operator*. This is the construction deferred from [§6.2](operators-spectral-theorem.ipynb).

### Projectors

For a normalized $|e\rangle$, the operator

```{math}
:label: eq-projector
P=|e\rangle\langle e|,\qquad P|\psi\rangle=\langle e|\psi\rangle\,|e\rangle,\qquad P^2=P,\quad P=P^{\dagger},\quad \text{eigenvalues }\{0,1\} ,
```

is the **projector** onto the $e$-direction: it keeps only the part of $|\psi\rangle$ along $|e
\rangle$. Projectors are **idempotent** ($P^2=P$ — projecting twice is projecting once), **Hermitian**,
and **rank-1**. Physically the projector is the mathematical content of a measurement (the postulate
is [§6.5](postulates.ipynb)).

### The resolution of the identity

For any orthonormal basis $\{|e_i\rangle\}$,

```{math}
:label: eq-dirac-resolution
\sum_i|e_i\rangle\langle e_i|=I,\qquad |\psi\rangle=\sum_i|e_i\rangle\langle e_i|\psi\rangle=\sum_i c_i|e_i\rangle,\quad c_i=\langle e_i|\psi\rangle .
```

This compact identity *is* "every state expands in the basis" — recovering the expansion
coefficients of [§6.1](complex-vector-spaces.ipynb), now as projections. The trick that makes the notation flow is to **insert the identity**
anywhere a basis change or expansion is wanted.

### Change of basis as a unitary

Changing from one orthonormal basis to another is a **unitary** $U$ whose columns are the new basis
vectors; components and operators transform as

```{math}
:label: eq-change-basis
\psi'=U^{\dagger}\psi,\qquad A'=U^{\dagger}AU,\qquad \text{eigenvalues and inner products preserved} .
```

A change of basis is a passive relabelling, not a physical change (cf. the unitary operators of [§6.2](operators-spectral-theorem.ipynb)).

### The spectral decomposition in projector form

The spectral theorem of [§6.2](operators-spectral-theorem.ipynb), $A=V\Lambda V^{\dagger}$, is exactly

```{math}
:label: eq-spectral-dirac
A=\sum_i\lambda_i\,|\lambda_i\rangle\langle\lambda_i|,\qquad P_iP_j=\delta_{ij}P_i,\quad \sum_iP_i=I ,
```

a Hermitian operator as its eigenvalues times the **projectors** onto its eigenvectors. The spectral
projectors are orthogonal and resolve the identity. This is the cleanest statement of "diagonalize,"
and the form the measurement postulate uses ([§6.5](postulates.ipynb)).

### Functions of operators

Because $A=\sum_i\lambda_i|\lambda_i\rangle\langle\lambda_i|$, *any* function of the operator is

```{math}
:label: eq-function-operator
f(A)=\sum_i f(\lambda_i)\,|\lambda_i\rangle\langle\lambda_i| ,
```

apply $f$ to the eigenvalues, in the eigenbasis. This is the general machinery behind the
exponential map of [§6.2](operators-spectral-theorem.ipynb), $e^{-iH}=\sum_i e^{-i\lambda_i}|\lambda_i\rangle\langle\lambda_i|$, and behind the
**time-evolution operator** $U(t)=e^{-iHt/\hbar}=\sum_n e^{-iE_nt/\hbar}|n\rangle\langle n|$ — the
single most important function of an operator in the volume ([§6.7](time-evolution.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: kets are 1-D complex arrays in a fixed orthonormal basis of ℂⁿ; a bra is the
# conjugate (row functional), so ⟨φ|ψ⟩ = bra(φ) @ ψ = numpy.vdot(φ, ψ) (the §6.1 convention). The outer
# product |u⟩⟨v| = numpy.outer(u, numpy.conj(v)) is an operator. We diagonalize with numpy.linalg.eigh.


def ket(components):
    """Construct a ket $|\\psi\\rangle$ from its components {eq}`eq-bra-ket`.

    A ket is a state vector — a 1-D complex array of components in a fixed orthonormal basis.

    Parameters
    ----------
    components : array_like
        The components $\\psi_i$ in the working basis.

    Returns
    -------
    numpy.ndarray
        The ket as a complex 1-D array.
    """
    return np.asarray(components, dtype=complex)


def bra(psi):
    """Return the bra $\\langle\\psi|$ dual to the ket ``psi`` {eq}`eq-bra-ket`.

    The bra is the linear functional "take the inner product with $\\psi$"; in components it is the
    conjugate (the adjoint row vector, ``psi.conj()``), and it acts on a ket by the matrix product
    ``bra(psi) @ phi`` $=\\langle\\psi|\\phi\\rangle=$ ``numpy.vdot(psi, phi)``.

    Parameters
    ----------
    psi : numpy.ndarray
        A ket (1-D complex array).

    Returns
    -------
    numpy.ndarray
        The conjugated components, to be applied to a ket with ``@``.
    """
    return psi.conj()


def outer(u, v):
    """The outer product $|u\\rangle\\langle v|$, an operator {eq}`eq-outer`.

    Built with ``numpy.outer(u, numpy.conj(v))`` (the bra conjugates ``v``). Acting on a ket it returns
    $\\langle v|w\\rangle\\,|u\\rangle$ — it measures the overlap with $v$ and outputs that much of $u$.
    Contrast the inner product $\\langle v|u\\rangle$, which is a number.

    Parameters
    ----------
    u, v : numpy.ndarray
        Kets (1-D complex arrays).

    Returns
    -------
    numpy.ndarray
        The operator $|u\\rangle\\langle v|$ as a 2-D array.
    """
    return np.outer(u, np.conj(v))


def projector(e):
    """The projector $P=|e\\rangle\\langle e|$ onto a normalized state ``e`` {eq}`eq-projector`.

    Equal to ``outer(e, e)``. It is idempotent ($P^2=P$), Hermitian, and rank-1, and $P|\\psi\\rangle=
    \\langle e|\\psi\\rangle|e\\rangle$ keeps only the component of $|\\psi\\rangle$ along $|e\\rangle$.
    Physically it is the mathematical content of a measurement outcome (the postulate is §6.5).

    Parameters
    ----------
    e : numpy.ndarray
        A normalized ket.

    Returns
    -------
    numpy.ndarray
        The rank-1 projector onto ``e``.
    """
    return np.outer(e, np.conj(e))


def resolution_of_identity(basis):
    """Sum the rank-1 projectors of an orthonormal basis, $\\sum_i|e_i\\rangle\\langle e_i|$ {eq}`eq-dirac-resolution`.

    For an orthonormal basis this sum equals the identity — the statement that every state expands in
    the basis. ``basis`` holds the basis vectors as rows; the function sums ``projector(e_i)`` over them.

    Parameters
    ----------
    basis : numpy.ndarray, shape (n, n)
        An orthonormal basis, one vector per row.

    Returns
    -------
    numpy.ndarray
        $\\sum_i|e_i\\rangle\\langle e_i|$, which is the identity for an orthonormal basis.
    """
    n = basis.shape[1]
    return sum(projector(basis[i]) for i in range(basis.shape[0])).reshape(n, n)


def function_of_operator(H, f):
    """Apply a scalar function ``f`` to a Hermitian operator via its spectral decomposition {eq}`eq-function-operator`.

    Diagonalizes ``H`` with ``numpy.linalg.eigh`` and returns $\\sum_i f(\\lambda_i)|\\lambda_i\\rangle
    \\langle\\lambda_i|$ — the function applied to the eigenvalues in the eigenbasis. This is the general
    machine behind the exponential map ($f=e^{-i\\,\\cdot}$) and the time-evolution operator (§6.7).

    Parameters
    ----------
    H : numpy.ndarray
        A Hermitian matrix.
    f : callable
        A scalar function applied elementwise to the eigenvalues.

    Returns
    -------
    numpy.ndarray
        The operator $f(H)$.
    """
    eigenvalues, V = np.linalg.eigh(H)
    return sum(f(eigenvalues[i]) * projector(V[:, i]) for i in range(len(eigenvalues)))

## Exercise 1 — Kets, bras, and the outer product

Let $|\phi\rangle=(1,\ i,\ 0)^{\mathsf T}$ and $|\psi\rangle=(0,\ 1,\ i)^{\mathsf T}$ and
$|w\rangle=(2,\ 0,\ 1)^{\mathsf T}$ in $\mathbb{C}^3$. Form the inner product $\langle\phi|\psi
\rangle$ and the outer product $|\phi\rangle\langle\psi|$, identify what *kind* of object each is,
and verify that the outer product acts as an operator via $\big(|u\rangle\langle
v|\big)|w\rangle=\langle v|w\rangle|u\rangle$ {eq}`eq-bra-ket`, {eq}`eq-outer`.

1. Build the kets with the `ket` helper (1-D complex arrays).
2. Form the inner product $\langle\phi|\psi\rangle$ with `bra(phi) @ psi` (equivalently
   `numpy.vdot(phi, psi)`) and confirm it is a **scalar** (`numpy.ndim` is 0).
3. Form the outer product $|\phi\rangle\langle\psi|$ with the `outer` helper (`numpy.outer(phi,
   numpy.conj(psi))`) and confirm it is an **operator** (a 2-D array, shape $3\times3$).
4. Verify the action: compute $\big(|\phi\rangle\langle\psi|\big)|w \rangle$ as `outer(phi, psi) @
   w` and compare with $\langle\psi|w\rangle|\phi\rangle$ (`numpy.vdot( psi, w) * phi`) using
   `numpy.allclose`.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    np.ndim(inner) == 0
    and Op.shape == (3, 3)
    and np.allclose(outer(phi, psi) @ w, np.vdot(psi, w) * phi),
    "the inner product ⟨φ|ψ⟩ is a number; the outer product |φ⟩⟨ψ| is an operator with (|φ⟩⟨ψ|)|w⟩ = ⟨ψ|w⟩|φ⟩",
)

## Exercise 2 — Projectors

For the normalized state $|e\rangle=(1,\ i,\ 1)^{\mathsf T}/\sqrt3$, build the projector
$P=|e\rangle\langle e|$ and verify its three defining properties — idempotence ($P^2=P$),
Hermiticity ($P=P^{\dagger}$), and rank-1 spectrum (eigenvalues $\{0,0,1\}$) — then show that
$P|\psi\rangle=\langle e|\psi\rangle|e\rangle$ extracts the $e$-component of a state
{eq}`eq-projector`.

1. Normalize $|e\rangle$ (`numpy.linalg.norm`) and build $P$ with the `projector` helper
   (`numpy.outer(e, numpy.conj(e))`).
2. Verify idempotence with `numpy.allclose(P @ P, P)`, Hermiticity with `numpy.allclose(P,
   P.conj().T)`, and the rank-1 spectrum with `numpy.linalg.eigvalsh` (eigenvalues sort to
   $\{0,0,1\}$).
3. Take an arbitrary state $|\psi\rangle$ and confirm `P @ psi` equals $\langle
   e|\psi\rangle|e\rangle$ (`numpy.vdot(e, psi) * e`). Frame: a projector is the mathematics of a
   measurement outcome ([§6.5](postulates.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    idempotent and hermitian and rank_one and extracts,
    "a projector P=|e⟩⟨e| is idempotent (P²=P), Hermitian, rank-1 (eigenvalues {0,1}), and extracts the e-component, P|ψ⟩=⟨e|ψ⟩|e⟩",
)

## Exercise 3 — The resolution of the identity

For an orthonormal basis $\{|e_i\rangle\}$ of $\mathbb{C}^4$, show that the sum of its rank-1
projectors is the identity, $\sum_i|e_i\rangle\langle e_i|=I$, and use this to expand an arbitrary
state as $|\psi\rangle=\sum_i\langle e_i|\psi\rangle|e_i\rangle$ {eq}`eq-dirac-resolution`.

1. Build an orthonormal basis by `numpy.linalg.qr` of a random complex matrix (its columns are
   orthonormal); take the columns as the basis vectors.
2. Sum the projectors with the `resolution_of_identity` helper and confirm the result equals
   `numpy.eye(4)` (`numpy.max(numpy.abs( R - numpy.eye(4)))`).
3. Expand a state by *inserting the identity*: compute $\sum_i\langle e_i|\psi \rangle|e_i\rangle$
   (a sum of `numpy.vdot(e_i, psi) * e_i`) and confirm with `numpy.allclose` that it reconstructs
   $|\psi\rangle$ — recovering the expansion coefficients of [§6.1](complex-vector-spaces.ipynb) as projections. Frame: "insert the
   identity" is the move that makes Dirac notation flow.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    resolution_error,
    0.0,
    "an orthonormal basis resolves the identity, Σ|e_i⟩⟨e_i| = I",
    atol=1e-12,
)
validate.check(
    np.allclose(sum(np.vdot(e_basis[i], psi3) * e_basis[i] for i in range(4)), psi3),
    "inserting the identity expands any state, |ψ⟩ = Σ⟨e_i|ψ⟩|e_i⟩ (the coefficients of §6.1 as projections)",
)

## Exercise 4 — Change of basis as a unitary

Take the Hermitian operator $H$ built below and the orthonormal basis $\{|e_i\rangle\}$ of
Exercise 3. Transform a state and the operator into that basis, show the basis-change matrix is
unitary, and confirm the transformation preserves eigenvalues and inner products — a passive
relabelling, not a physical change {eq}`eq-change-basis`.

1. Assemble the basis-change matrix $U$ with `numpy.column_stack` so its columns are the new basis
   vectors.
2. Verify $U^{\dagger}U=I$ with `numpy.allclose(U.conj().T @ U, numpy.eye(4))`.
3. Transform the state as $\psi'=U^{\dagger}\psi$ (`U.conj().T @ psi`) and the operator as
   $A'=U^{\dagger}AU$ (`U.conj().T @ H @ U`).
4. Verify the eigenvalues are unchanged (`numpy.linalg.eigvalsh` of $H$ and of $A'$ agree) and an
   inner product is preserved ($\langle U^{\dagger}u|U^{\dagger}v\rangle=\langle u|v\rangle$).
   Frame: a change of basis is a unitary ([§6.2](operators-spectral-theorem.ipynb)), physically inert.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    np.allclose(U.conj().T @ U, np.eye(4)) and eig_preserved and inner_preserved,
    "a change of basis is a unitary transformation (U†U=I) that preserves eigenvalues and inner products — a passive relabelling",
)

## Exercise 5 — The spectral decomposition in projector form

For the Hermitian operator $H$ of Exercise 4, write it as a sum of eigenvalues times the
projectors onto its eigenvectors, $H=\sum_i\lambda_i|\lambda_i\rangle\langle\lambda_i|$, and
verify the reconstruction together with the orthogonality and completeness of the spectral
projectors {eq}`eq-spectral-dirac`.

1. Diagonalize $H$ with `numpy.linalg.eigh`, giving eigenvalues $\lambda_i$ and the eigenvectors
   as the columns of $V$.
2. Build the spectral projectors $P_i=|\lambda_i\rangle\langle \lambda_i|$ with the `projector`
   helper on each column `V[:, i]`.
3. Reconstruct $\sum_i\lambda_iP_i$ (a sum of `lambda_i * P_i`) and compare with $H$ via
   `numpy.max(numpy.abs(H - sum))`.
4. Verify the projectors are orthogonal ($P_iP_j=0$ for $i\neq j$, via `numpy.allclose(P_i @ P_j,
   0)`) and resolve the identity ($\sum_iP_i=I$). Frame: this is "diagonalize" in its cleanest
   form — the form measurement uses ([§6.5](postulates.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    spectral_error,
    0.0,
    "the spectral decomposition A = Σλ_i|λ_i⟩⟨λ_i| writes a Hermitian operator via its eigen-projectors",
    atol=1e-12,
)
validate.check(
    orthogonal < 1e-10 and complete,
    "the spectral projectors are orthogonal (P_iP_j=0, i≠j) and resolve the identity (ΣP_i=I)",
)

## Exercise 6 — Functions of operators

Define a function of a Hermitian operator through its spectral decomposition, $f(A)=\sum_i
f(\lambda_i)|\lambda_i\rangle\langle\lambda_i|$, and verify it on two functions: the exponential
$e^{-iH}$ (cross-checked against `scipy.linalg.expm`) and the square root $\sqrt{H_+}$ of a
positive operator (which must square back to $H_+$) {eq}`eq-function-operator`.

1. Implement $f(A)$ with the `function_of_operator` helper, which diagonalizes with
   `numpy.linalg.eigh` and sums $f(\lambda_i)\,$`projector(V[:, i])`.
2. Build $e^{-iH}$ as `function_of_operator(H, lambda x: numpy.exp(-1j * x))` and compare it to
   the *true* matrix exponential `scipy.linalg.expm(-1j * H)` (`numpy.max(numpy.abs(...))`) — they
   must agree.
3. Build a positive operator $H_+=HH^{\dagger}+I$, take $\sqrt{H_+}$ as
   `function_of_operator(H_plus, numpy.sqrt)`, and confirm it squares back: `sqrtH @ sqrtH` equals
   $H_+$ (`numpy.allclose`). Frame: any function of an operator is $f$ applied to its eigenvalues;
   the time-evolution operator $e^{-iHt/\hbar}$ ([§6.7](time-evolution.ipynb)) is the prime example.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [exp_match, sqrt_squared_error],
    [0.0, 0.0],
    "a function of an operator is f applied to its eigenvalues, f(A)=Σf(λ_i)|λ_i⟩⟨λ_i| (e^{-iH} matches scipy.linalg.expm; √(H₊) squares back)",
    atol=1e-10,
)

## Exercise 7 — The qubit in two bases, and a projector measurement

Take the qubit state $|\psi\rangle=\cos\tfrac{\theta}{2}|0\rangle+e^{i\varphi}\sin
\tfrac{\theta}{2}|1\rangle$ with $\theta=1.4,\ \varphi=1.1$. Express it in the computational ($z$)
basis $\{|0\rangle,|1\rangle\}$ and the ($x$) basis $\{|{+}\rangle,|{-}\rangle\}$, relate the two
by a unitary, verify each basis resolves the identity, and compute the projector weights $|\langle
e|\psi \rangle|^2$ in each — confirming they sum to one in *both* {eq}`eq-change-basis`,
{eq}`eq-projector`, {eq}`eq-dirac-resolution`.

1. Build $|\psi\rangle$ with the `ket` helper and both bases ($|{\pm}\rangle=(|0\rangle
   \pm|1\rangle)/\sqrt2$).
2. Form the Hadamard-like unitary $U$ connecting them with `numpy.column_stack` of
   $|{+}\rangle,|{-}\rangle$, and confirm `is`-unitary via `numpy.allclose(U.conj().T @ U,
   numpy.eye(2))`.
3. Verify each basis resolves the identity with the `resolution_of_identity` helper (both equal
   `numpy.eye(2)`).
4. Compute the projector weights $|\langle e|\psi\rangle|^2$ in each basis (`abs(numpy.vdot(e,
   psi))**2`) and confirm each set sums to 1 (`numpy.sum`). Frame: the same state, two bases, two
   sets of outcome-probabilities — a direct preview of measurement in different bases ([§6.5](postulates.ipynb)) and of
   the Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    z_resolves
    and x_resolves
    and np.isclose(z_weights.sum(), 1.0)
    and np.isclose(x_weights.sum(), 1.0),
    "a state has well-defined outcome probabilities in any orthonormal basis: both bases resolve the identity and their projector weights sum to 1",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The calculus of states and operators

Dirac notation turned the inner product into a **bra acting on a ket**, the outer product into an
**operator**, and three constructions into the working calculus of quantum mechanics. With the
**projector** $|e\rangle\langle e|$ we extract a component; with the **resolution of the identity**
$\sum_i|e_i\rangle\langle e_i|=I$ we expand in any basis at will — *insert the identity*; with the
**spectral decomposition** $A=\sum_i\lambda_i|\lambda_i\rangle\langle\lambda_i|$ we diagonalize; and
with $f(A)=\sum_i f(\lambda_i)|\lambda_i\rangle\langle\lambda_i|$ we apply any function to an operator
by applying it to the eigenvalues. The arsenal is complete: we have the **space** ([§6.1](complex-vector-spaces.ipynb)), the
**operators** ([§6.2](operators-spectral-theorem.ipynb)), and now the **language** (§6.3). Movement 0 is closed.

**This exercise (synthesis).** No new computation: the calculus is the result. We have spent three
notebooks building it and have not yet measured anything — and yet the apparatus of measurement is
already in our hands. The projector we built *is* a measurement outcome; the resolution of the
identity *is* the set of possible outcomes; the spectral decomposition *is* the observable, its
eigenvalues the results and its projectors the outcomes. The physics was hiding in the notation. The
next notebook ([§6.4](stern-gerlach-qubit.ipynb)) lets it out, taking this entire apparatus to the Stern–Gerlach experiment and the
two-state system — the first real measurement of the volume, and the start of Movement I.

## Notebook summary

Dirac notation, the language of the rest of the volume, and the close of Movement 0.

- **Kets, bras, the dual space** {eq}`eq-bra-ket`: a ket is a state; a bra $\langle\phi|=$`phi.conj()`
  is the linear functional "inner product with $\phi$"; $\langle\phi|\psi\rangle$ is a number.
- **The outer product** {eq}`eq-outer`: $|u\rangle\langle v|=$`numpy.outer(u, numpy.conj(v))` is an
  *operator*, $\big(|u\rangle\langle v|\big)|w\rangle=\langle v|w\rangle|u\rangle$.
- **Projectors** {eq}`eq-projector`: $P=|e\rangle\langle e|$ is idempotent, Hermitian, rank-1, and
  extracts the $e$-component — the mathematics of a measurement ([§6.5](postulates.ipynb)).
- **The resolution of the identity** {eq}`eq-dirac-resolution`: $\sum_i|e_i\rangle\langle e_i|=I$ — *insert
  the identity* to expand in any basis; recovers the coefficients of [§6.1](complex-vector-spaces.ipynb) as projections.
- **Change of basis** {eq}`eq-change-basis`: a unitary $A'=U^{\dagger}AU$, $\psi'=U^{\dagger}\psi$,
  preserving eigenvalues and inner products — a passive relabelling.
- **The spectral decomposition** {eq}`eq-spectral-dirac`: $A=\sum_i\lambda_i|\lambda_i\rangle\langle
  \lambda_i|$, the eigen-projectors orthogonal and complete — "diagonalize" in its cleanest form.
- **Functions of operators** {eq}`eq-function-operator`: $f(A)=\sum_i f(\lambda_i)|\lambda_i\rangle
  \langle\lambda_i|$; $e^{-iH}$ matches `scipy.linalg.expm`, the seed of time evolution ([§6.7](time-evolution.ipynb)).

The arsenal is complete — the space, the operators, the language. The projector is already a
measurement, the resolution of the identity already the set of outcomes, the spectral decomposition
already the observable. The physics was hiding in the notation; Movement I lets it out.

## Outlook

- **The postulates of quantum mechanics ([§6.5](postulates.ipynb))**, written in this language: observables as Hermitian
  operators, outcomes as eigenvalues, probabilities as $|\langle\lambda|\psi\rangle|^2$, measurement
  as projection.
- **The Stern–Gerlach experiment and the two-state system ([§6.4](stern-gerlach-qubit.ipynb))**: the formalism meets a real
  measurement, and Movement I begins.
- **The time-evolution operator ([§6.7](time-evolution.ipynb))**: $U(t)=e^{-iHt/\hbar}=\sum_n e^{-iE_nt/\hbar}|n\rangle\langle
  n|$, the prime function of an operator.
- **Cross-reference** [§6.1](complex-vector-spaces.ipynb) (the inner product, expansion coefficients), [§6.2](operators-spectral-theorem.ipynb) (the spectral theorem, the
  exponential map), and forward to [§6.4](stern-gerlach-qubit.ipynb), [§6.5](postulates.ipynb), [§6.7](time-evolution.ipynb), [§6.8](bloch-sphere-entanglement.ipynb).

In [ ]:
from ecp.style import footer

footer()